In [14]:
import duckdb

con = duckdb.connect('your_database.duckdb')

print(con.execute('''
SELECT
    AVG(total_shots) AS mean_shots,
    MEDIAN(total_shots) AS median_shots,
    STDDEV(total_shots) AS std_shots,
    MIN(total_shots) AS min_shots,
    MAX(total_shots) AS max_shots
FROM player_season
''').fetchdf())

   mean_shots  median_shots  std_shots  min_shots  max_shots
0   17.340046          10.0   22.81106        0.0      228.0


In [15]:
query = """
SELECT
    player_name,
    total_shots,
    (total_shots - AVG(total_shots) OVER ()) / STDDEV(total_shots) OVER () AS z_score,
    PERCENT_RANK() OVER (ORDER BY total_shots) AS percentile_rank
FROM player_season
WHERE total_shots > 0
ORDER BY total_shots DESC
LIMIT 10
"""

print(con.execute(query).fetchdf())

                           player_name  total_shots   z_score  percentile_rank
0  Cristiano Ronaldo dos Santos Aveiro        228.0  8.801683         1.000000
1              Gonzalo Gerardo Higuaín        182.0  6.845953         0.999446
2       Lionel Andrés Messi Cuccittini        158.0  5.825571         0.998338
3                           Harry Kane        158.0  5.825571         0.998338
4                   Zlatan Ibrahimović        148.0  5.400412         0.997784
5                          Andy Delort        143.0  5.187833         0.997230
6             Luis Alberto Suárez Díaz        139.0  5.017769         0.996676
7                      Lorenzo Insigne        135.0  4.847706         0.996122
8                           Paul Pogba        124.0  4.380031         0.995568
9                Michy Batshuayi Tunga        123.0  4.337515         0.994460


In [16]:
query = """
SELECT
    *,
    (total_shots / total_minutes) * 90 AS shots_per_90,
    (total_passes / total_minutes) * 90 AS passes_per_90
FROM player_season
ORDER BY total_minutes DESC
LIMIT 10
"""
print(con.execute(query).fetchdf())

   player_id              player_name  total_minutes  matches_played  \
0       3815        Kasper Schmeichel         3581.0              38   
1       3813               Wes Morgan         3581.0              38   
2       3608            Simon Francis         3578.0              38   
3       3344            Andrew Surman         3578.0              38   
4      20005        Toby Alderweireld         3558.0              38   
5       3742         Orestis Karnezis         3550.0              38   
6       4506           Nampalys Mendy         3542.0              38   
7       3522  Heurelho da Silva Gomes         3542.0              38   
8       3523             Craig Dawson         3538.0              38   
9       6378                Jan Oblak         3535.0              38   

   total_shots  total_passes  shots_per_90  passes_per_90  
0          0.0        1220.0      0.000000      30.661826  
1         21.0         849.0      0.527786      21.337615  
2          6.0        2454.

In [17]:
query = """
SELECT
    CASE
        WHEN total_minutes < 500 THEN '<500'
        WHEN total_minutes < 900 THEN '500-899'
        WHEN total_minutes < 1500 THEN '900-1499'
        WHEN total_minutes < 2500 THEN '1500-2499'
        ELSE '2500+'
    END AS minutes_bucket,
    COUNT(*) AS num_players
FROM player_season
GROUP BY minutes_bucket
ORDER BY MIN(total_minutes)
"""
print(con.execute(query).fetchdf())

  minutes_bucket  num_players
0           <500          563
1        500-899          264
2       900-1499          363
3      1500-2499          566
4          2500+          429


In [18]:
con.execute("""
CREATE OR REPLACE TABLE player_season AS
SELECT
    *,
    (total_shots / total_minutes) * 90 AS shots_per_90,
    (total_passes / total_minutes) * 90 AS passes_per_90
FROM player_season
""")

print(con.execute("SELECT * FROM player_season LIMIT 5").fetchdf())

   player_id                player_name  total_minutes  matches_played  \
0       5485             Raphaël Varane         2263.0              26   
1       7900  Tiago Manuel Dias Correia         2288.0              34   
2       6832        Rubén Castro Martín         3481.0              38   
3       3677      Bakary Adama Soumaoro         2426.0              28   
4     401509          David Ducourtioux         2657.0              32   

   total_shots  total_passes  shots_per_90  passes_per_90  
0         12.0        1258.0      0.477243      50.030932  
1         80.0         842.0      3.146853      33.120629  
2        103.0        1022.0      2.663028      26.423442  
3         13.0         980.0      0.482275      36.356142  
4         30.0        1530.0      1.016184      51.825367  


In [19]:
query = """
SELECT
    CASE
        WHEN total_minutes < 500 THEN '<500'
        WHEN total_minutes < 900 THEN '500-899'
        WHEN total_minutes < 1500 THEN '900-1499'
        WHEN total_minutes < 2500 THEN '1500-2499'
        ELSE '2500+'
    END AS minutes_bucket,
    STDDEV(shots_per_90) AS std_shots_per_90
FROM player_season
GROUP BY minutes_bucket
ORDER BY MIN(total_minutes)
"""
print(con.execute(query).fetchdf())

  minutes_bucket  std_shots_per_90
0           <500          1.673031
1        500-899          0.880622
2       900-1499          0.959666
3      1500-2499          0.955512
4          2500+          1.059798
